In [1]:
import abc
import random as rd
import typing as tp

import numpy as np

In [2]:
class Arm(tp.Protocol):
    def pull(self) -> int:
        pass

    def get_prob(self) -> float:
        pass

In [3]:
class BanditStrategy(tp.Protocol):
    def get_counts(self) -> list[float]:
        pass

    def get_values(self) -> list[float]:
        pass

    def get_total_reward(self) -> int:
        pass

    def update(self, n_arm: int, reward: int) -> None:
        pass

    def choose(self) -> int:
        pass

In [4]:
class SlotMachine(Arm):
    def __init__(self, name: str, probability: float) -> None:
        self._name = name
        self._probability = probability

    def pull(self) -> int:
        return 1 if rd.random() < self._probability else 0

    def get_prob(self) -> float:
        return self._probability

    def __str__(self) -> str:
        return self._name

In [5]:
class BaseBanditStrategy(BanditStrategy):
    def __init__(self, n_arms: int) -> None:
        self._n_arms = n_arms
        self._counts = np.zeros(self._n_arms)
        self._values = np.zeros(self._n_arms)
        self._total_reward = 0

    def get_counts(self) -> list[float]:
        return self._counts.tolist()

    def get_values(self) -> list[float]:
        return self._values.tolist()

    def get_total_reward(self) -> int:
        return self._total_reward

    def update(self, n_arm: int, reward: int) -> None:
        self._update_counts(n_arm)
        self._update_values(n_arm, reward)
        self._update_cum_reward(reward)

    @abc.abstractmethod
    def choose(self) -> int:
        pass

    def _update_values(self, n_arm: int, reward: int) -> None:
        self._values[n_arm] += (reward - self._values[n_arm]) / self._counts[n_arm]

    def _update_counts(self, n_arm: int) -> None:
        self._counts[n_arm] += 1

    def _update_cum_reward(self, reward: int) -> None:
        self._total_reward += reward

    def __str__(self) -> str:
        return self.__class__.__name__

In [6]:
class RandomBanditStrategy(BaseBanditStrategy):
    def choose(self) -> int:
        return np.random.randint(self._n_arms)


In [7]:
class GreedyBanditStrategy(BaseBanditStrategy):
    def choose(self) -> int:
        if np.min(self._counts) == 0:
            return int(np.argmin(self._counts))

        return int(np.argmax(self._values))

In [8]:
class EpsilonRandomGreedyBanditStrategy(BaseBanditStrategy):
    def __init__(self, n_arms: int, epsilon: float) -> None:
        super().__init__(n_arms)
        self._epsilon = epsilon

    def choose(self) -> int:
        if np.min(self._counts) == 0:
            return int(np.argmin(self._counts))

        if np.random.random() < self._epsilon:
            return int(np.random.randint(self._n_arms))
        return int(np.argmax(self._values))

In [9]:
def test_bandit_strategy(strategy: BanditStrategy, arms: tp.Sequence[Arm], attempts: int) -> None:
    for _ in range(attempts):
        n_arm = strategy.choose()
        reward = arms[n_arm].pull()
        strategy.update(n_arm, reward)

In [10]:
def calculate_regret_perc(strategy: BanditStrategy, arms: tp.Sequence[Arm], attempts: int) -> float:
    best_prob = max(arm.get_prob() for arm in arms)
    if best_prob == 0:
        return 0
    best_total_reward = best_prob * attempts
    actual_total_reward = strategy.get_total_reward()
    regret = best_total_reward - actual_total_reward
    return regret / best_total_reward

In [11]:
def print_results(strategy: BanditStrategy, arms: tp.Sequence[Arm], regret: float) -> None:
    values = strategy.get_values()
    print(strategy)
    print("Arm | Prob Value %")
    for i, arm in enumerate(arms):
        print(f"{str(arm):^3} | {values[i]:.2f}")
    print(f"Regret: {regret:.2f}")
    print()

In [12]:
arms = [
    SlotMachine("A", 0.1),
    SlotMachine("B", 0.2),
    SlotMachine("C", 0.3),
]
n_arms = len(arms)
attempts = 1000
epsilon = 0.1
strategies = [
    RandomBanditStrategy(n_arms),
    GreedyBanditStrategy(n_arms),
    EpsilonRandomGreedyBanditStrategy(n_arms, epsilon),
]

for strategy in strategies:
    test_bandit_strategy(strategy, arms, attempts)
    regret = calculate_regret_perc(strategy, arms, attempts)
    print_results(strategy, arms, regret)
    calculate_regret_perc(strategy, arms, attempts)

RandomBanditStrategy
Arm | Prob Value %
 A  | 0.10
 B  | 0.21
 C  | 0.31
Regret: 0.30

GreedyBanditStrategy
Arm | Prob Value %
 A  | 0.09
 B  | 0.00
 C  | 0.00
Regret: 0.69

EpsilonRandomGreedyBanditStrategy
Arm | Prob Value %
 A  | 0.10
 B  | 0.21
 C  | 0.30
Regret: 0.04



## Выбор статического Epsilon
Epsilon -> 1.00 когда (мало попыток) + (варианты близки по вероятностям) + (вероятности динамическиe)


Epsilon -> 0.00 когда (много попыток) + (варианты различны по вероятностям) + (вероятности статичны)

## UCB (Upper Confidence Bound)
Даем каждому варианту **бонус за неопределенность**. Чем меньше мы его пробовали - тем больше бонус. Потом выбираем вариант с максимальной оценкой + бонус.
```
choose(n_arm) = среднее_вознаграждение + √(2 × ln(total_attempts) / n_arm_attempts)
                 ↑                          ↑
            эксплуатация              бонус за исследование
```

Где:
- `total_attempts` = общее кол-во потраченных попыток
- `n_arm_attempts` = кол-во попыток потраченных на нынешний вариант


In [13]:
class UCBBanditStrategy(BaseBanditStrategy):
    def choose(self) -> int:
        if np.min(self._counts) == 0:
            return int(np.argmin(self._counts))

        total_counts = np.sum(self._counts)
        ucb_values = self._values + np.sqrt(2 * np.log(total_counts) / self._counts)
        return int(np.argmax(ucb_values))

In [14]:
arms = [
    SlotMachine("A", 0.1),
    SlotMachine("B", 0.2),
    SlotMachine("C", 0.3),
    SlotMachine("D", 0.6),
]
n_arms = len(arms)
attempts = 10000
epsilon = 0.1
strategies = [
    RandomBanditStrategy(n_arms),
    GreedyBanditStrategy(n_arms),
    EpsilonRandomGreedyBanditStrategy(n_arms, epsilon),
    UCBBanditStrategy(n_arms)
]

for strategy in strategies:
    test_bandit_strategy(strategy, arms, attempts)
    regret = calculate_regret_perc(strategy, arms, attempts)
    print_results(strategy, arms, regret)
    calculate_regret_perc(strategy, arms, attempts)

RandomBanditStrategy
Arm | Prob Value %
 A  | 0.10
 B  | 0.20
 C  | 0.28
 D  | 0.58
Regret: 0.52

GreedyBanditStrategy
Arm | Prob Value %
 A  | 0.00
 B  | 0.00
 C  | 0.00
 D  | 0.60
Regret: 0.00

EpsilonRandomGreedyBanditStrategy
Arm | Prob Value %
 A  | 0.12
 B  | 0.18
 C  | 0.28
 D  | 0.60
Regret: 0.05

UCBBanditStrategy
Arm | Prob Value %
 A  | 0.07
 B  | 0.22
 C  | 0.29
 D  | 0.60
Regret: 0.02



## Thompson Sampling (Байесовский подход)
Главная проблема предыдущих подходов в их **детерминированности**, всегда выбирают один и тот же вариант в одинаковой ситуации.
Из-за этого мы не можем учитывать неопределенность, как пример:
Представь, что ты пробуешь три ресторана:

**Стандартный подход:**
- Вариант A: 0.2
- Вариант B: 0.5 <- Выбираем!
- Вариант C: 0.3

**Байесовский подход:**
- Вариант A: потратили 100 попыток → считаем что вероятность 0.2 ± 10%
- Вариант B: потратили 5 попыток → считаем что вероятность 0.5 ± 80%
- Вариант C: потратили 50 попыток → считаем что вероятность 0.3 ± 30%

**Вопрос:** Какому результату доверяем больше?

**Главная идея байесовского подхода** - Вместо того чтобы хранить детерминант, мы можем хранить распределение вероятностей.

Для бинарных наград используется **бета-распределение**.

Бета-распределение описывается двумя параметрами: Beta(**α**, **β**), где: α = количество успехов + 1, β = количество провалов + 1.
Наиболее вероятностная оценка: α/(α+β)
Выбираем вариант с наибольшим сэмплом.

In [15]:
class ThompsonSamplingBanditStrategy(BaseBanditStrategy):
    def __init__(self, n_arms: int) -> None:
        super().__init__(n_arms)
        self._alpha = np.ones(self._n_arms)
        self._beta = np.ones(self._n_arms)

    def choose(self) -> int:
        theta = np.random.beta(self._alpha, self._beta)
        return int(np.argmax(theta))

    def update(self, n_arm: int, reward: int) -> None:
        super().update(n_arm, reward)
        self._update_beta_and_alpha(n_arm, reward)

    def _update_beta_and_alpha(self, n_arm: int, reward: int) -> None:
        if reward == 1:
            self._alpha[n_arm] += 1
        else:
            self._beta[n_arm] += 1

In [16]:
arms = [
    SlotMachine("A", 0.1),
    SlotMachine("B", 0.2),
    SlotMachine("C", 0.3),
    SlotMachine("D", 0.6),
    SlotMachine("E", 0.5),
]
n_arms = len(arms)
attempts = 10000
epsilon = 0.1
strategies = [
    RandomBanditStrategy(n_arms),
    GreedyBanditStrategy(n_arms),
    EpsilonRandomGreedyBanditStrategy(n_arms, epsilon),
    UCBBanditStrategy(n_arms),
    ThompsonSamplingBanditStrategy(n_arms),
]

for strategy in strategies:
    test_bandit_strategy(strategy, arms, attempts)
    regret = calculate_regret_perc(strategy, arms, attempts)
    print_results(strategy, arms, regret)
    calculate_regret_perc(strategy, arms, attempts)

RandomBanditStrategy
Arm | Prob Value %
 A  | 0.09
 B  | 0.20
 C  | 0.30
 D  | 0.61
 E  | 0.50
Regret: 0.43

GreedyBanditStrategy
Arm | Prob Value %
 A  | 0.50
 B  | 0.00
 C  | 0.00
 D  | 0.60
 E  | 0.57
Regret: -0.00

EpsilonRandomGreedyBanditStrategy
Arm | Prob Value %
 A  | 0.09
 B  | 0.21
 C  | 0.34
 D  | 0.60
 E  | 0.56
Regret: 0.05

UCBBanditStrategy
Arm | Prob Value %
 A  | 0.08
 B  | 0.17
 C  | 0.26
 D  | 0.60
 E  | 0.45
Regret: 0.03

ThompsonSamplingBanditStrategy
Arm | Prob Value %
 A  | 0.15
 B  | 0.30
 C  | 0.14
 D  | 0.60
 E  | 0.49
Regret: 0.00

